# Bratislava Precipitation Forecast Browser

Interactive visualization of ICON-D2-RUC-EPS precipitation forecasts for Bratislava.

**Features:**
- Interactive selection of statistical variables (mean, median, percentiles)
- Multi-run forecast comparison
- Timestamp-based visualization (run time + forecast hour)
- Uncertainty bands and ensemble analysis

**Data Source:** Processed by `bratislava_pipeline.py`

In [ ]:
# Setup and imports
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Interactive widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Note: ipywidgets not available - using static plots")

print("✓ Imports loaded")
print(f"  Interactive widgets: {WIDGETS_AVAILABLE}")
print(f"  Bratislava coordinates: 48.1486°N, 17.1077°E")

In [ ]:
# Load Bratislava precipitation data
print("📂 Loading Bratislava precipitation data...")

data_dir = Path("data/bratislava")
data_files = sorted(list(data_dir.glob("bratislava_precipitation_*.nc")))

if not data_files:
    print("❌ No Bratislava data found!")
    print("   Run 'python bratislava_pipeline.py' first")
    data_loaded = False
    data = None
else:
    # Load the most recent file
    latest_file = max(data_files)
    
    try:
        data = xr.open_dataset(latest_file)
        print(f"✅ Loaded: {latest_file.name}")
        
        # Display data info
        print(f"   Location: {data.attrs.get('coordinates', 'Unknown')}")
        print(f"   Creation: {data.attrs.get('creation_date', 'Unknown')}")
        print(f"   Data shape: {data.tp.shape}")
        print(f"   Dimensions: {dict(data.dims)}")
        
        # Show run times and forecast range
        if 'run_time' in data.dims:
            run_times = pd.to_datetime(data.run_time.values)
            print(f"   Forecast runs: {len(run_times)}")
            for i, rt in enumerate(run_times):
                print(f"     {i+1}: {rt.strftime('%Y-%m-%d %H:%M')}")
        
        if 'step' in data.dims:
            steps = data.step.values
            print(f"   Forecast range: {steps[0]:.0f} to {steps[-1]:.0f} hours ({len(steps)} steps)")
        
        # Show available variables
        stat_vars = [var for var in data.data_vars if var.startswith('tp_')]
        print(f"   Statistical variables: {len(stat_vars)}")
        for var in sorted(stat_vars):
            print(f"     • {var}")
        
        data_loaded = True
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        data_loaded = False
        data = None

In [ ]:
# Configuration for visualization
if data_loaded:
    # Available statistical variables
    stat_vars = [var for var in data.data_vars if var.startswith('tp_')]
    
    AVAILABLE_VARIABLES = {
        'tp_mean': 'Ensemble Mean',
        'tp_median': 'Ensemble Median',
        'tp_p05': '5th Percentile',
        'tp_p10': '10th Percentile', 
        'tp_p25': '25th Percentile',
        'tp_p50': '50th Percentile (Median)',
        'tp_p75': '75th Percentile',
        'tp_p90': '90th Percentile',
        'tp_p95': '95th Percentile',
        'tp_min': 'Ensemble Minimum',
        'tp_max': 'Ensemble Maximum',
        'tp_std': 'Standard Deviation'
    }
    
    # Keep only variables that exist in the data
    AVAILABLE_VARIABLES = {k: v for k, v in AVAILABLE_VARIABLES.items() if k in stat_vars}
    
    print(f"Available variables for visualization: {list(AVAILABLE_VARIABLES.keys())}")
else:
    AVAILABLE_VARIABLES = {}

## Interactive Precipitation Browser

In [ ]:
# Interactive visualization dashboard for Bratislava
if WIDGETS_AVAILABLE and data_loaded:
    print("🎛️ Setting up interactive Bratislava precipitation browser...")
    
    # Get forecast run information
    run_times = pd.to_datetime(data.run_time.values)
    run_options = [(rt.strftime('%Y-%m-%d %H:%M'), i) for i, rt in enumerate(run_times)]
    
    # Create interactive widgets
    variable_widget = widgets.Dropdown(
        options=[(desc, var) for var, desc in AVAILABLE_VARIABLES.items()],
        value=list(AVAILABLE_VARIABLES.keys())[0],
        description='Statistical Variable:',
        style={'description_width': '130px'},
        layout={'width': '400px'}
    )
    
    run_widget = widgets.SelectMultiple(
        options=run_options,
        value=list(range(len(run_options))),  # Select all runs by default
        description='Forecast Runs:',
        style={'description_width': '130px'},
        layout={'width': '300px', 'height': '100px'}
    )
    
    show_uncertainty_widget = widgets.Checkbox(
        value=True,
        description='Show uncertainty bands',
        style={'description_width': 'initial'}
    )
    
    # Output widgets for charts
    chart1_output = widgets.Output(layout={'height': '500px'})
    chart2_output = widgets.Output(layout={'height': '800px'})
    
    def create_timestamps(run_time, forecast_hours):
        """Create timestamps by adding forecast hours to run time"""
        run_time_dt = pd.to_datetime(run_time)
        timestamps = [run_time_dt + pd.Timedelta(hours=float(h)) for h in forecast_hours]
        return timestamps
    
    def create_combined_chart(selected_var, selected_runs):
        """Create combined chart showing all selected runs"""
        
        fig = go.Figure()
        colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
        
        for i, run_idx in enumerate(selected_runs):
            run_time = pd.to_datetime(data.run_time.values[run_idx])
            run_data = data.isel(run_time=run_idx)
            
            # Create timestamps
            timestamps = create_timestamps(run_time, run_data.step.values)
            
            # Get variable data
            var_data = run_data[selected_var]
            
            fig.add_trace(go.Scatter(
                x=timestamps,
                y=var_data.values,
                mode='lines+markers',
                name=f"Run {run_time.strftime('%H:%M')}",
                line=dict(color=colors[i % len(colors)], width=2),
                marker=dict(size=6),
                hovertemplate=(
                    f"<b>Run {run_time.strftime('%H:%M')}</b><br>" +
                    "Time: %{x}<br>" +
                    "Precipitation: %{y:.3f} mm/h<br>" +
                    "<extra></extra>"
                )
            ))
        
        fig.update_layout(
            title=f"{AVAILABLE_VARIABLES[selected_var]} - Bratislava<br>All Selected Forecast Runs",
            xaxis_title="Time",
            yaxis_title="Precipitation (mm/h)",
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=12),
            height=450,
            xaxis=dict(
                tickformat='%H:%M<br>%m/%d',
                dtick=3600000 * 3,  # 3 hour intervals
            )
        )
        
        return fig
    
    def create_multi_panel_chart(selected_var, selected_runs, show_uncertainty):
        """Create multi-panel chart with uncertainty ranges per run"""
        
        n_runs = len(selected_runs)
        if n_runs == 1:
            rows, cols = 1, 1
        elif n_runs <= 2:
            rows, cols = 1, 2
        elif n_runs <= 4:
            rows, cols = 2, 2
        else:
            rows, cols = 3, 2
        
        fig = make_subplots(
            rows=rows, cols=cols,
            subplot_titles=[f"Run {pd.to_datetime(data.run_time.values[run_idx]).strftime('%H:%M')}" 
                          for run_idx in selected_runs],
            shared_yaxes=True,
            shared_xaxes=True
        )
        
        for i, run_idx in enumerate(selected_runs):
            row = i // cols + 1
            col = i % cols + 1
            
            run_time = pd.to_datetime(data.run_time.values[run_idx])
            run_data = data.isel(run_time=run_idx)
            
            # Create timestamps
            timestamps = create_timestamps(run_time, run_data.step.values)
            
            # Main variable
            main_var = run_data[selected_var]
            
            # Add uncertainty bands if requested
            if show_uncertainty:
                # Try to get uncertainty bounds
                lower_var = None
                upper_var = None
                
                if 'tp_p10' in run_data and 'tp_p90' in run_data:
                    lower_var = run_data['tp_p10']
                    upper_var = run_data['tp_p90']
                    uncertainty_label = "10th-90th percentile range"
                elif 'tp_p25' in run_data and 'tp_p75' in run_data:
                    lower_var = run_data['tp_p25']
                    upper_var = run_data['tp_p75']
                    uncertainty_label = "25th-75th percentile range"
                elif 'tp_min' in run_data and 'tp_max' in run_data:
                    lower_var = run_data['tp_min']
                    upper_var = run_data['tp_max']
                    uncertainty_label = "Min-max range"
                
                if lower_var is not None and upper_var is not None:
                    # Add uncertainty band
                    fig.add_trace(go.Scatter(
                        x=timestamps,
                        y=upper_var.values,
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False,
                        hoverinfo='skip'
                    ), row=row, col=col)
                    
                    fig.add_trace(go.Scatter(
                        x=timestamps,
                        y=lower_var.values,
                        mode='lines',
                        fill='tonexty',
                        fillcolor='rgba(0,100,80,0.2)',
                        line=dict(width=0),
                        name=uncertainty_label if i == 0 else "",
                        showlegend=(i == 0),
                        hoverinfo='skip'
                    ), row=row, col=col)
            
            # Add main line
            fig.add_trace(go.Scatter(
                x=timestamps,
                y=main_var.values,
                mode='lines+markers',
                name=AVAILABLE_VARIABLES[selected_var] if i == 0 else "",
                showlegend=(i == 0),
                line=dict(color='blue', width=2),
                marker=dict(size=4),
                hovertemplate=(
                    f"<b>{AVAILABLE_VARIABLES[selected_var]}</b><br>" +
                    "Time: %{x}<br>" +
                    "Precipitation: %{y:.3f} mm/h<br>" +
                    "<extra></extra>"
                )
            ), row=row, col=col)
        
        fig.update_layout(
            title=f"{AVAILABLE_VARIABLES[selected_var]} - Bratislava<br>Individual Forecast Runs with Uncertainty",
            height=700,
            template='plotly_white',
            font=dict(size=11)
        )
        
        # Update axes
        fig.update_xaxes(
            title_text="Time", 
            tickformat='%H:%M<br>%m/%d',
            dtick=3600000 * 3  # 3 hour intervals
        )
        fig.update_yaxes(title_text="Precipitation (mm/h)", col=1)
        
        return fig
    
    def update_visualizations(*args):
        """Update both visualizations"""
        try:
            selected_var = variable_widget.value
            selected_runs = list(run_widget.value)
            show_uncertainty = show_uncertainty_widget.value
            
            if not selected_runs:
                with chart1_output:
                    clear_output()
                    print("⚠️ Please select at least one forecast run")
                with chart2_output:
                    clear_output()
                return
            
            # Chart 1: Combined view
            with chart1_output:
                clear_output(wait=True)
                print(f"🎨 Generating combined chart for Bratislava...")
                fig1 = create_combined_chart(selected_var, selected_runs)
                fig1.show()
            
            # Chart 2: Multi-panel view
            with chart2_output:
                clear_output(wait=True)
                print(f"🎨 Generating multi-panel chart for Bratislava...")
                fig2 = create_multi_panel_chart(selected_var, selected_runs, show_uncertainty)
                fig2.show()
                
        except Exception as e:
            with chart1_output:
                clear_output()
                print(f"❌ Error creating combined chart: {e}")
            with chart2_output:
                clear_output()
                print(f"❌ Error creating multi-panel chart: {e}")
    
    # Connect widgets
    variable_widget.observe(update_visualizations, 'value')
    run_widget.observe(update_visualizations, 'value')
    show_uncertainty_widget.observe(update_visualizations, 'value')
    
    # Create dashboard layout
    controls_box = widgets.VBox([
        widgets.HTML("<h3>🌧️ Bratislava Precipitation Forecast Browser</h3>"),
        widgets.HTML(f"<p>Location: {data.attrs.get('coordinates', 'Bratislava, Slovakia')}</p>"),
        widgets.HTML("<p>Select statistical variable and forecast runs to update the charts.</p>"),
        widgets.HBox([variable_widget]),
        widgets.HBox([run_widget, show_uncertainty_widget]),
        widgets.HTML("<hr>")
    ])
    
    charts_box = widgets.VBox([
        widgets.HTML("<h4>📊 Chart 1: Combined View - All Selected Runs</h4>"),
        chart1_output,
        widgets.HTML("<h4>📈 Chart 2: Multi-Panel View - Individual Runs with Uncertainty</h4>"),
        chart2_output
    ])
    
    dashboard = widgets.VBox([controls_box, charts_box])
    
    # Display dashboard
    display(dashboard)
    
    # Generate initial charts
    print("\nGenerating initial visualizations...")
    update_visualizations()
    
    print("✅ Interactive Bratislava browser ready!")
    print("\nHow to use:")
    print("• Select a statistical variable (mean, median, percentiles)")
    print("• Choose one or more forecast runs (hold Ctrl/Cmd for multiple)")
    print("• Toggle uncertainty bands on/off")
    print("• Charts show actual timestamps (run time + forecast hours)")
    print("• Hover over points for detailed information")
    
elif data_loaded:
    print("📊 Interactive widgets not available - creating static example")
    
    # Create a simple static plot
    if 'tp_mean' in data.data_vars:
        fig = go.Figure()
        
        colors = ['blue', 'red', 'green', 'orange']
        
        for i, run_time in enumerate(pd.to_datetime(data.run_time.values)):
            run_data = data.isel(run_time=i)
            timestamps = [run_time + pd.Timedelta(hours=float(h)) for h in run_data.step.values]
            
            fig.add_trace(go.Scatter(
                x=timestamps,
                y=run_data.tp_mean.values,
                mode='lines+markers',
                name=f"Run {run_time.strftime('%H:%M')}",
                line=dict(color=colors[i % len(colors)], width=2)
            ))
        
        fig.update_layout(
            title="Bratislava Ensemble Mean Precipitation",
            xaxis_title="Time",
            yaxis_title="Precipitation (mm/h)",
            template='plotly_white',
            height=400
        )
        
        fig.show()
        print("✅ Static example chart displayed")

else:
    print("❌ No data loaded - cannot create visualization")
    print("\nTo get started:")
    print("1. Run 'python bratislava_pipeline.py' to download and process data")
    print("2. Re-run this notebook to visualize the results")

## Data Summary

In [ ]:
# Display data summary and statistics
if data_loaded:
    print("📊 Bratislava Precipitation Data Summary")
    print("=" * 45)
    
    # Basic information
    print(f"Location: {data.attrs.get('coordinates', 'Bratislava, Slovakia')}")
    print(f"Data source: {data.attrs.get('source', 'ICON-D2-RUC-EPS')}")
    print(f"Processing: {data.attrs.get('processing', 'Unknown')}")
    print(f"Created: {data.attrs.get('creation_date', 'Unknown')}")
    
    # Dimensions
    print(f"\nData dimensions:")
    for dim_name, dim_size in data.dims.items():
        print(f"  {dim_name}: {dim_size}")
    
    # Forecast information
    if 'run_time' in data.dims:
        run_times = pd.to_datetime(data.run_time.values)
        print(f"\nForecast runs: {len(run_times)}")
        for i, rt in enumerate(run_times):
            print(f"  {i+1}: {rt.strftime('%Y-%m-%d %H:%M UTC')}")
    
    if 'step' in data.dims:
        steps = data.step.values
        print(f"\nForecast range: {steps[0]:.0f} to {steps[-1]:.0f} hours")
        print(f"Time steps: {len(steps)}")
        
        # Calculate typical interval
        if len(steps) > 1:
            intervals = np.diff(steps)
            typical_interval = float(np.median(intervals))
            print(f"Typical interval: {typical_interval:.2f} hours ({typical_interval*60:.0f} minutes)")
    
    # Statistical variables
    stat_vars = [var for var in data.data_vars if var.startswith('tp_')]
    print(f"\nStatistical variables: {len(stat_vars)}")
    for var in sorted(stat_vars):
        var_data = data[var]
        min_val = float(var_data.min())
        max_val = float(var_data.max())
        mean_val = float(var_data.mean())
        print(f"  • {var}: {min_val:.3f} - {max_val:.3f} mm/h (avg: {mean_val:.3f})")
    
    # Precipitation analysis
    if 'tp' in data.data_vars:
        tp_data = data.tp
        total_precip = float(tp_data.sum())
        max_rate = float(tp_data.max())
        non_zero = (tp_data > 0.001).sum()
        total_points = tp_data.size
        
        print(f"\nPrecipitation analysis:")
        print(f"  Maximum rate: {max_rate:.3f} mm/h")
        print(f"  Non-zero values: {non_zero} / {total_points} ({100*non_zero/total_points:.1f}%)")
    
    print(f"\n✅ Use the interactive browser above to explore the forecasts!")
else:
    print("❌ No data available for summary")

In [ ]:
# Clean up resources
print("🧹 Browser session complete")
print(f"Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Close dataset to free memory
if 'data' in globals() and data is not None:
    try:
        data.close()
    except:
        pass

print("✅ Memory cleanup completed")
print("\n💡 To refresh data, run 'python bratislava_pipeline.py' again")